# Seq2seq model and attention

## Table of Content
1. Seq2seq model
2. Seq2seq with attention


Complete the following codes to accomplish two neural networks: **Seq2seq** model and **Seq2seq with attention**. We will use machine translation as an example task to testify the performance of the two models.

> Machine Translation is to generate a sentence in a target language for a sentence in another source language. That is, both the inputs and outputs of the model are a sequence of words, which is the same as Seq2seq model.

Here are data information today for the two neural models.

*  Training corpus: many pairs of parallel sentences in two different languages.

> You can choose the language pairs of interests and download the txt file from http://www.manythings.org/anki/. Here, we choose  Mandarin Chinese to English as an example. Please rename and place it at "/content/drive/MyDrive/SMU_MITB_NLP/lab4/cmn.txt".

Below we have provides some basic codes to prepare your data or your model, but you are free to use your own.

It is also recommended to request a **GPU** for training.

In [ ]:
#@title show your CPU or GPU details
from tensorflow.python.client import device_lib
device_lib.list_local_devices()

In [ ]:
#@title connect google drive folder

from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/SMU_MITB_NLP/lab4/

In [ ]:
#@title import useful packages
import gensim
import re
import math
import numpy as np
import torch
from torch.autograd import Variable
import torch.nn as nn
import torch.nn.functional as F
import random

from torch.utils.data import (DataLoader, RandomSampler, SequentialSampler,
                              TensorDataset)
import pickle
from queue import PriorityQueue
import operator

In [ ]:
#@title Utility functions for reading and preprocessing data

MAX_LEN = 15

def read_corpus(file_name):
    with open(file_name, "r", encoding="utf-8") as fin:
        for line in fin:
            line = line.strip()
            items = line.split('\t')
            if len(items) < 2: continue
            en_sent = preprocess(items[0], lang='en')
            zh_sent = preprocess(items[1], lang='zh')
            if len(en_sent) > MAX_LEN or len(zh_sent) > MAX_LEN: continue
            yield en_sent, zh_sent

def preprocess(sent, lang):
    tokens = []
    if lang == 'en':
        tokens = gensim.utils.simple_preprocess(sent, min_len=1)
    elif lang == 'zh':
        tokens = [w for w in list(re.sub('[\W]+', '', sent))]
    return tokens

In [ ]:
clean_data = list(read_corpus("./cmn.txt"))
print("The number of parallel pairs: ", len(clean_data))
print(clean_data[:2])

## Data preprocessing for NN

In [ ]:
#@title class PreProcessor

class Lang():

    def __init__(self, name, pad_token_id=0, beg_token_id=1, end_token_id=2, unk_token_id=3):
        self.name = name
        self.w2id_vocab = {'<PAD>':pad_token_id, '<CLS>':beg_token_id, '<SEP>':end_token_id, '<UNK>':unk_token_id}
        self.id2w_vocab = {pad_token_id:'<PAD>', beg_token_id:'<CLS>', end_token_id:'<SEP>', unk_token_id:'<UNK>'}
        self.pad_token_id = pad_token_id
        self.beg_token_id = beg_token_id
        self.end_token_id = end_token_id
        self.unk_token_id = unk_token_id

        self.count_vocab = {}
        self.vocab_size = len(self.w2id_vocab)
        self.max_seq_length = 0

    def get_ids_from_tokens(self, words):
        return [self.w2id_vocab.get(t, self.unk_token_id) for t in words]

    def get_words_from_ids(self, wids):
        return [self.id2w_vocab.get(t, self.id2w_vocab[self.unk_token_id]) for t in wids if t != self.pad_token_id]

    def add_tokens(self, tokens, max_len=64):
        if len(tokens) > self.max_seq_length:
            self.max_seq_length = len(tokens)
        if len(tokens) <= max_len-2:
            for t in tokens:
                count = self.count_vocab.get(t, 0)
                count += 1
                self.count_vocab[t] = count

    def build_vocab(self, mincount=1):
        for t in self.count_vocab:
            if self.count_vocab[t] < mincount : continue
            if t not in self.w2id_vocab:
                self.w2id_vocab[t] = self.vocab_size
                self.id2w_vocab[self.vocab_size] = t
                self.vocab_size += 1

class PreProcessor():

    def __init__(self, lang1, lang2):

        self.pad_token_id = 0
        self.beg_token_id = 1
        self.end_token_id = 2
        self.unk_token_id = 3

        self.lang1 = Lang(lang1, self.pad_token_id, self.beg_token_id, self.end_token_id, self.unk_token_id)
        self.lang2 = Lang(lang2, self.pad_token_id, self.beg_token_id, self.end_token_id, self.unk_token_id)

        self.model_args = {
            # data preprocessing parameters
            'max_vocabulary_size': 50000,
            'min_occurrence': 1,  # Remove all words that does not appears at least n times.
            'max_seq_length': MAX_LEN,  # make all sentence the same length for batching
            # Training Parameters.
            'hidden_size': 256,
            'learning_rate': 0.001,
            'batch_size': 256,
            'n_layers': 2,
            'n_epochs': 500,
            'clip': 5,         # clip the gradients in case of too large value
            'init_seed': 3407,
            'teacher_forcing_ratio': 0.5,
            'dropout': 0.1,
            # log parameter
            'verbose': True,  # if output log info
            'log_step': 30,
            'checkpoint_path': "./seq2seq.bin",
        }

    def save(self, path):
        f = open(path, 'wb')
        pickle.dump(self, f)
        f.close()

    def load(self, path):
        f = open(path, 'rb')
        proc = pickle.load(f)
        f.close()
        return proc

    def set_model_arg(self, key, value):
        self.model_args[key] = value

    def get_model_arg(self, key):
        return self.model_args.get(key, None)

    def build_vocab(self, data):
        # build vocabulary
        for en_tokens, zh_tokens in data:
            self.lang1.add_tokens(en_tokens, self.get_model_arg('max_seq_length'))
            self.lang2.add_tokens(zh_tokens, self.get_model_arg('max_seq_length'))
        # sort vocab according to their frequency
        self.lang1.build_vocab(self.get_model_arg('min_occurrence'))
        self.lang2.build_vocab(self.get_model_arg('min_occurrence'))

    def convert_examples_to_features(self, examples):
        # process sents: a list of strings
        features = []

        for (ex_index, example) in enumerate(examples):

            if len(example[0]) > self.get_model_arg('max_seq_length') or len(example[1]) > self.get_model_arg('max_seq_length')-1:
                continue
            # use to build mask for batching
            lang1_input_ids = self.convert_sentence_to_features(example[0], self.lang1.name)
            lang2_input_ids = self.convert_sentence_to_features(example[1], self.lang2.name)

            if ex_index < 5 and self.get_model_arg('verbose'):
                print("*** Example ***")
                print("lang1_input_ids: %s" % " ".join([str(x) for x in lang1_input_ids]))
                print("lang2_input_ids: %s" % " ".join([str(x) for x in lang2_input_ids]))

            features.append((lang1_input_ids, lang2_input_ids))
        return features

    def convert_sentence_to_features(self, sent, lang):
        # Creating lists that will hold our input and target sequences
        # Remove last token and convert to ids for input sequence

        if lang == self.lang1.name:
            input_seq = self.lang1.get_ids_from_tokens(sent)
        else:
            input_seq = self.lang2.get_ids_from_tokens(sent) + [self.end_token_id]
        input_ids = self.padding_sent(input_seq, max_length=self.get_model_arg('max_seq_length'))

        return tuple(input_ids)

    # output tuple based ids
    def padding_sent(self, input_ids, max_length=512):
        # Zero-pad up to the sequence length.
        padding_length = max_length - len(input_ids)

        input_ids = input_ids + ([self.pad_token_id] * padding_length)

        assert len(input_ids) == max_length, "Error with input length {} vs {}".format(len(input_ids), max_length)

        return input_ids


    def convert_feature_to_dataset(self, features):

        lang1_input_ids = torch.tensor([f[0] for f in features], dtype=torch.long)
        lang2_input_ids = torch.tensor([f[1] for f in features], dtype=torch.long)

        return TensorDataset(lang1_input_ids, lang2_input_ids)

    def get_data_iter(self, features, batch_size, drop_last=True):
        dataset = self.convert_feature_to_dataset(features)

        dataset_sampler = RandomSampler(dataset)    # SequentialSampler(dataset)

        dataloader = DataLoader(dataset, sampler=dataset_sampler, batch_size=batch_size, drop_last=drop_last)

        return iter(dataloader)

In [ ]:
#@title prepare dataset and hyper-parameters for training
proc = PreProcessor('en', 'zh')

# hyper-parameters for data
proc.build_vocab(clean_data)


print("{} vocab size: {}, {} vocab size: {}".format(proc.lang1.name, proc.lang1.vocab_size, proc.lang2.name, proc.lang2.vocab_size))
print("{} max seq length: {}, {} max seq length: {}".format(proc.lang1.name, proc.lang1.max_seq_length, proc.lang2.name, proc.lang2.max_seq_length))
# save proc
arg_path = "./proc.dat"
proc.save(arg_path)

In [ ]:
#@title test your batched dataset
features = proc.convert_examples_to_features(clean_data)
data_iter = proc.get_data_iter(features, proc.get_model_arg('batch_size'))
tmp_batch = next(data_iter)   # each batch is a list of two tensors, where each tensor: batch_size * max_seq_length
print("lang1 input: ", proc.lang1.get_words_from_ids(tmp_batch[0].numpy()[0]))
print("lang2 input: ", proc.lang2.get_words_from_ids(tmp_batch[1].numpy()[0]))

In [ ]:
#@title Note that changing the tensor shape with or without transpose may lead to different tensor!
src = tmp_batch[0]
print(src)    # bs * seq
print(src.view(-1))

In [ ]:
trg = src.transpose(0, 1).contiguous()
print(trg)    # seq * bs
print(trg.view(-1))

## Seq2seq model
A seq2seq model consists of two components:

* Encoder is to learn context representation c as inputs to decoder.
* Decoder is to make predictions conditioned on the given contexts.

A simple architecture based on two RNNs shown in the below figure.

![](https://drive.google.com/uc?export=view&id=1Orflav1ei3ZHlk21lQe8VJEi6-l2EMx7)

Now, let's finish the following classes of **Encoder**, **VanillaDecoder**, and **Seq2seq**, inherit from *torch.nn.Module*, to build a seq2seq model (without attention).

**Hint**: You can use pytorch packages (e.g., nn.gru), instead our own implementation. More details please refer to the pytorch documentation [link](https://pytorch.org/docs/stable/index.html).


### Getting familiar with torch.nn.GRU

[nn.GRU](https://pytorch.org/docs/stable/generated/torch.nn.GRU.html) is a pytorch implementation of GRU.

#### Inputs:


*   inputs: tensor shape: (seq_length, batch_size, hidden_size), which is the default case --- (batch_first = False) when you create your GRU.
*   hidden: $h_0$ in our slides, tensor shape: (num_direction*num_layers, batch_size, hidden_size)

#### outputs:


*   output: the hidden states of the last layer (including all timesteps). Tensor shape: (seq_length, batch_size, num_direction*hidden_size)
*   hidden: the hidden states of all the layers (only including the last timestep). Tensor shape: (num_direction*num_layers, batch_size, hidden_size)

Here is an example of LSTM illustration to clarify the inputs and outputs.
![](https://drive.google.com/uc?export=view&id=1Y8J15ZUX-RKvSgnwVdiVKfAHxRxwXgCO)

Note that the two outputs (output and hidden) may has some overlaps. You can use GRU in three ways as follows and observe the outputs.


In [ ]:
#@title example inputs
torch.manual_seed(1)
# Parameters
input_size = 3  # Input feature size
hidden_size = 3  # Hidden state size
seq_length = 4  # Number of timesteps in the sequence
batch_size = 2  # Number of samples in a batch

# Initialize the GRU layer
num_layers = 1    # Number of stacked GRU layers
tmp_gru = nn.GRU(input_size, hidden_size, num_layers, bidirectional=False)

# Create sample input data (seq_length x batch_size x input_size)
tmp_inputs = torch.randn(seq_length, batch_size, input_size)
print('input: ', tmp_inputs)

In [ ]:
#@title 1) conduct RNN per time step per batch sample

# Initialize hidden state (num_layers * num_directions, batch_size, hidden_size)
hidden = torch.zeros(num_layers, 1, hidden_size)

for ti in tmp_inputs:   # loop for timestep
    for bi in ti:   # loop for each batch sample
        # bi.shape : (input_size), so we need to change the shape to (1, 1, input_size), (1, 1) denotes per timestep per batch sample
        # hidden is shared by different batch samples, leading to different outputs from the following two ways
        out, hidden = tmp_gru(bi.view(1, 1, -1), hidden)
        print('out:', out)
        print('hidden:', hidden)

In [ ]:
#@title 2) conduct RNN for all batch samples per time step

# Initialize hidden state (num_layers * num_directions, batch_size, hidden_size)
hidden = torch.zeros(num_layers, batch_size, hidden_size)

for ti in tmp_inputs:
    out, hidden = tmp_gru(ti.view(1, batch_size, -1), hidden)
    print('out:', out)
    print('hidden:', hidden)

In [ ]:
#@title 3) conduct rnn forward pass in one time

# Initialize hidden state (num_layers * num_directions, batch_size, hidden_size)
hidden = torch.zeros(num_layers, batch_size, hidden_size)

out, hidden = tmp_gru(tmp_inputs, hidden)
print('out:', out)
print('hidden:', hidden)

In [ ]:
#@title encoder pytorch implementation

class Encoder(nn.Module):
    # dropout is a technique to improve the model generalization ability, model details can be found
    # https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html
    def __init__(self, vocab_size, hidden_size, n_layers=1, dropout=0.1):
        super(Encoder, self).__init__()

        # do not change the code above
        # write your code here

        # do not change the code below

    # Note: we run this all at once (over multiple batches of multiple sequences), like the above method 3.
    def forward(self, inputs, hidden=None):
        # inputs: [seq_len, batch_size]
        # do not change the code above
        # write your code here

        # do not change the code below
        # outputs: [seq_len, batch_size, hidden_size]
        # hidden: [n_layers, batch_size, hidden_size]
        return outputs, hidden

In [ ]:
#@title test encoder!

hidden_size = proc.get_model_arg('hidden_size')
n_layers = proc.get_model_arg('n_layers')
# Instantiate the model with hyperparameters
tmp_encoder = Encoder(proc.lang1.vocab_size, hidden_size, n_layers)
# tmp_batch[0]: [batch_size, max_len]
tmp_outputs, tmp_hidden = tmp_encoder(tmp_batch[0].transpose(0, 1))

print('encoder_outputs', tmp_outputs.size()) # [max_len, batch_size, hidden_size]
print('encoder_hidden', tmp_hidden.size()) # [n_layers, batch_size, hidden_size]

In [ ]:
#@title decoder pytorch implementation without attention
class VanillaDecoder(nn.Module):

    def __init__(self, vocab_size, hidden_size, n_layers=1, dropout=0.1):
        super(VanillaDecoder, self).__init__()
        # do not change the code above
        # write your code here

        # do not change the code below

    # Note: we run this one step at a time, like the above method 2.
    def forward(self, input, hidden, encoder_hiddens=None):
        # input = [batch_size]
        # hidden = [n_layers, batch_size, hidden_size]
        # encoder_hiddens, this won't be used here, just keep consistent with attention decoder

        # do not change the code above
        # write your code here

        # do not change the code below
        # output: [batch_size, vocab_size]
        return output, hidden

In [ ]:
#@title test vanilla decoder!
tmp_decoder = VanillaDecoder(proc.lang2.vocab_size, hidden_size, n_layers)

batch_size = proc.get_model_arg('batch_size')
hidden = torch.zeros(n_layers, batch_size, hidden_size)

# tmp_batch[1]: [batch_size, max_len]
for i in range(2):
    tmp_outputs, tmp_hidden = tmp_decoder(tmp_batch[1].transpose(0, 1)[i], hidden)

    print('decoder_outputs', tmp_outputs.size()) # [batch_size, vocab_size]
    print('decoder_hidden', tmp_hidden.size()) # [n_layers, batch_size, hidden_size]

In [ ]:
#@title seq2seq by combinbing encoder and decoder

class Seq2Seq(nn.Module):

    def __init__(self, encoder, decoder):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder

        assert encoder.hidden_size == decoder.hidden_size, \
            "Hidden dimensions of encoder and decoder must be equal!"
        assert encoder.n_layers == decoder.n_layers, \
            "Number of layers of encoder and decoder must be equal!"

    def forward(self, src, trg, teacher_forcing_ratio = 0.5, start_token_id = 1):
        # src = [src_seq_len, batch_size]
        # trg = [trg_seq_len, batch_size]
        # teacher_forcing_ratio is probability to use teacher forcing (use ground truth, rather than predictions, for next timestamp predictions)
        # e.g. if teacher_forcing_ratio is 0.75 we use ground-truth inputs 75% of the time

        src_len, batch_size = src.shape
        trg_len, _ = trg.shape
        src_vocab_size = self.encoder.vocab_size
        trg_vocab_size = self.decoder.vocab_size

        # tensor to store encoder and decoder outputs
        trg_outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(trg.device)

        # encoding
        encoder_outputs, encoder_hidden = self.encoder(src)

        # decoding
        decoder_input = torch.LongTensor([start_token_id]*batch_size).to(trg.device)
        decoder_hidden = encoder_hidden

        use_teacher_forcing = True if random.random() < teacher_forcing_ratio else False

        if use_teacher_forcing:
            # Teacher forcing: Feed the target as the next input
            for di in range(trg_len):
                decoder_output, decoder_hidden = self.decoder(
                    decoder_input, decoder_hidden, encoder_outputs)
                decoder_input = trg[di]  # Teacher forcing
                trg_outputs[di] = decoder_output

        else:
            # Without teacher forcing: use its own predictions as the next input
            for di in range(trg_len):
                decoder_output, decoder_hidden = self.decoder(
                    decoder_input, decoder_hidden, encoder_outputs)
                topv, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze().detach()  # detach from history as input
                trg_outputs[di] = decoder_output

        # trg_outputs: [max_len, batch_size, vocab_size]
        return trg_outputs

In [ ]:
#@title test seq2seq!
tmp_model = Seq2Seq(tmp_encoder, tmp_decoder)

tmp_outputs = tmp_model(tmp_batch[0].transpose(0, 1), tmp_batch[1].transpose(0, 1))
print('seq2seq_outputs', tmp_outputs.size()) # [batch_size, vocab_size]

## Seq2seq with attention
Attention is a useful technique of neural networks to select informative features for a wide range of applications, such as question answering and information extraction.

The difference between vanilla seq2seq model and seq2seq with attention model lies in the decoder, shown in the below figure.

![](https://drive.google.com/uc?export=view&id=1Orflav1ei3ZHlk21lQe8VJEi6-l2EMx7)

At each timestamp of decoder, attention always involves:

1.   Computing the attention scores with all encoder hidden states;
2.   Taking softmax to get attention distribution;
3.   Using attention distribution to take weighted sum of encoder hidden states.

Finally, we cancatenate the attention output with the decoder hidden state to make prediction.

Now, let's revise the above decoder class for **seq2seq with attention** model. Encoder class and seq2seq class can be reused.



In [ ]:
#@title decoder with dot attention

class Decoder(nn.Module):

    def __init__(self, vocab_size, hidden_size, n_layers=1, dropout=0.1):
        super(Decoder, self).__init__()
        # do not change the code above
        # write your code here

        # do not change the code below

    def att(self, decoder_hidden, encoder_hiddens):
        # decoder_hidden =  [1, batch_size, hidden_size]
        # encoder_hiddens = [seq_len, batch_size, hidden_size]

        # do not change the code above
        # write your code here

        # do not change the code below
        # att_h = [batch_size, hidden_size*2]
        return att_h

    # Note: we run this one step at a time
    def forward(self, input, hidden, encoder_hiddens = None):
        # input = [batch_size]
        # hidden = [n_layers, batch_size, hidden_size]
        # encoder_hiddens = [seq_len, batch_size, hidden_size]

        # do not change the code above
        # write your code here

        # do not change the code below

        # output = [batch_size, vocab_size]
        return output, hidden


In [ ]:
#@title test attention decoder!
tmp_decoder = Decoder(proc.lang2.vocab_size, hidden_size, n_layers)

batch_size = proc.get_model_arg('batch_size')
max_seq_length = proc.get_model_arg('max_seq_length')
hidden = torch.zeros(n_layers, batch_size, hidden_size)
encoder_hidden = torch.zeros(max_seq_length, batch_size, hidden_size)

# tmp_batch[1]: [batch_size, max_len]
for i in range(2):
    tmp_outputs, tmp_hidden = tmp_decoder(tmp_batch[1].transpose(0, 1)[i], hidden, encoder_hidden)

    print('decoder_outputs', tmp_outputs.size()) # [batch_size, vocab_size]
    print('decoder_hidden', tmp_hidden.size()) # [n_layers, batch_size, hidden_size]

In [ ]:
#@title (extra) decoder with multiplicative attention

class Decoder(nn.Module):

    def __init__(self, vocab_size, hidden_size, n_layers=1, dropout=0.1):
        super(Decoder, self).__init__()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        self.dropout = nn.Dropout(dropout)

        self.emb = nn.Embedding(vocab_size, hidden_size, padding_idx=0)
        self.att_layer = nn.Linear(self.hidden_size, hidden_size)

        self.rnn = nn.GRU(hidden_size, hidden_size, n_layers, dropout = dropout)

        self.fc_out = nn.Linear(2*hidden_size, vocab_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def att(self, decoder_hidden, encoder_hiddens):
        # decoder_hidden =  [1, batch_size, hidden_size]
        # encoder_hiddens = [seq_len, batch_size, hidden_size]

        max_len, bs, _ = encoder_hiddens.size()

        # tmp_encoder_h = [batch_size, seq_len, hidden_size]
        tmp_encoder_h = encoder_hiddens.transpose(0, 1)

        # att_weights = [batch_size, 1, seq_len]
        tmp_encoder_h = self.att_layer(tmp_encoder_h)
        att_weights = torch.bmm(decoder_hidden.squeeze(0).unsqueeze(1), tmp_encoder_h.transpose(1, 2))
        att_weights = F.softmax(att_weights, dim=2)

        # c_v = [batch_size, hidden_size]
        c_v = torch.bmm(att_weights, tmp_encoder_h).squeeze(1)

        att_h = torch.cat( (c_v, decoder_hidden.squeeze(0)), dim=1 )
        # att_h = [batch_size, hidden_size*2]
        return att_h

    # Note: we run this one step at a time
    def forward(self, input, hidden, encoder_hiddens = None):
        # input = [batch_size]
        # hidden = [n_layers, batch_size, hidden_size]
        # encoder_hiddens = [seq_len, batch_size, hidden_size]


        bs = input.size(0)
        # x = [1, batch_size, hidden_size]
        x = self.dropout(self.emb(input)).view(1, bs, self.hidden_size)

        # Get current hidden state from input word and last hidden state
        # they have the same shape: [1, batch_size, hidden_size]
        rnn_output, hidden = self.rnn(x, hidden)

        att_h = self.att(rnn_output, encoder_hiddens)


        output = self.softmax(self.fc_out(att_h))
        # output = [batch_size, vocab_size]
        return output, hidden


In [ ]:
#@title test multiplicative attention decoder!
tmp_decoder = Decoder(proc.lang2.vocab_size, hidden_size, n_layers)

batch_size = proc.get_model_arg('batch_size')
max_seq_length = proc.get_model_arg('max_seq_length')
hidden = torch.zeros(n_layers, batch_size, hidden_size)
encoder_hidden = torch.zeros(max_seq_length, batch_size, hidden_size)

# tmp_batch[1]: [batch_size, max_len]
for i in range(2):
    tmp_outputs, tmp_hidden = tmp_decoder(tmp_batch[1].transpose(0, 1)[i], hidden, encoder_hidden)

    print('decoder_outputs', tmp_outputs.size()) # [batch_size, vocab_size]
    print('decoder_hidden', tmp_hidden.size()) # [n_layers, batch_size, hidden_size]

## Decoding

To generate texts, we have introduced two decoding methods that can take a sequence of tokens as inputs, and generate another sequence of tokens according the prediction probability over the vocab.

1. greedy decoding (to complish): at each time step, decode the token with the highest prediction probability.
2. beam search decoding (bonus): at each time step, decode topk tokens whose path from beginning till so far has the highest rewards.

In [ ]:
#@title greedy decoding

def greedy_translate(model, proc, max_len, input_sent, device):
    model.eval() # eval mode
    # model: our seq2seq model
    # proc: our preprocessor
    # input sent: a sequence of src tokens, e.g., "how are you?"
    # device: 'cpu' or 'cuda'
    # do not change the code above
    # write your code here

    # do not change the code below
    # decoded_sent: a sequence of target tokens, e.g., "你好吗？"
    return decoded_sent

In [ ]:
#@title (extra) beam search decoding

class BeamSearchNode(object):
    def __init__(self, hidden, previousNode, wordId, logProb, length):
        '''
        :param hiddenstate:
        :param previousNode:
        :param wordId:
        :param logProb:
        :param length:
        '''
        self.h = hidden
        self.prevNode = previousNode
        self.wordid = wordId
        self.logp = logProb
        self.leng = length

    def eval(self, alpha=1.0):
        reward = 0
        # Add here a function for shaping a reward

        return self.logp / float(self.leng - 1 + 1e-6) + alpha * reward

    def __lt__(self, other):
        return self.leng < other.leng

    def __gt__(self, other):
        return self.leng > other.leng

def beam_decode(model, proc, out_len, encoder_outputs, hidden):
    '''
    :param decoder_hidden: input tensor of shape [1, B, H] for start of the decoding
    :param encoder_outputs: if you are using attention mechanism you can pass encoder outputs, [T, B, H] where T is the maximum length of input sentence
    :return: decoded_batch
    '''

    beam_width = proc.get_model_arg('beam_width')
    topk = proc.get_model_arg('beam_topk')  # how many sentence do you want to generate

    if beam_width is None:
        beam_width = 10
    if topk is None:
        topk = 1
    # Start with the start of the sentence token
    decoder_input = torch.tensor([proc.beg_token_id], dtype=torch.long).to(device)

    # Number of sentence to generate
    endnodes = []
    number_required = min((topk + 1), topk - len(endnodes))

    # starting node -  hidden vector, previous node, word id, logp, length
    node = BeamSearchNode(hidden, None, decoder_input, 0, 1)
    nodes = PriorityQueue()

    # start the queue
    nodes.put((-node.eval(), node))
    qsize = 1

    # start beam search
    while True:
        # give up when decoding takes too long
        if qsize > 2000: break

        # fetch the best node
        score, n = nodes.get()
        decoder_input = n.wordid
        decoder_hidden = n.h

        if n.wordid.item() == proc.end_token_id and n.prevNode != None:
            endnodes.append((score, n))
            # if we reached maximum # of sentences required
            if len(endnodes) >= number_required:
                break
            else:
                continue

        # decode for one step using decoder
        output, hidden = model.decoder(decoder_input, hidden, encoder_outputs)
        # output.shape: 1 * vocab_size
        # hidden.shape: n_layer * 1 * vocab_size
        # PUT HERE REAL BEAM SEARCH OF TOP
        log_prob, indexes = torch.topk(output, beam_width)
        nextnodes = []
        # log_prob.shape: 1 * beam_width
        # indexes.shape: 1 * beam_width
        for new_k in range(beam_width):
            decoded_t = indexes[0][new_k].view(-1)
            log_p = log_prob[0][new_k].item()
            node = BeamSearchNode(hidden, n, decoded_t, n.logp + log_p, n.leng + 1)
            score = -node.eval()
            nextnodes.append((score, node))

        # put them into queue
        for i in range(len(nextnodes)):
            score, nn = nextnodes[i]
            nodes.put((score, nn))
            # increase qsize
        qsize += len(nextnodes) - 1

    # choose nbest paths, back trace them
    if len(endnodes) == 0:
        endnodes = [nodes.get() for _ in range(topk)]

    utterances = []
    for score, n in sorted(endnodes, key=operator.itemgetter(0)):
        utterance = []
        utterance.append(n.wordid.item())
        # back trace
        while n.prevNode != None:
            n = n.prevNode
            utterance.append(n.wordid.item())

        utterance = utterance[::-1]
        utterances.append(utterance[:out_len])

    return utterances

def beam_translate(model, proc, out_len, input_sent, device):
    # model: our seq2seq model
    # proc: our preprocessor
    # input sent: a sequence of src tokens, e.g., "how are you?"
    # device: 'cpu' or 'cuda'
    model.eval() # eval mode
    tokens = preprocess(input_sent, lang='en')
    input_seq = proc.lang1.get_ids_from_tokens(tokens)
    # inputs for encoder: [seq_length, batch_size]
    input_ids = torch.tensor(input_seq, dtype=torch.long).unsqueeze(1)
    encoder_outputs, hidden = model.encoder(input_ids.to(device))
    decoded_batch = beam_decode(model, proc, out_len, encoder_outputs, hidden)
    decoded_sents = [' '.join(proc.lang2.get_words_from_ids(sent)) for sent in decoded_batch]
    # decoded_sents: a list of sequences of target tokens, e.g., ["你好吗？", "你怎么样？"]
    return decoded_sents

## Training Seq2seq for MT

We have provided some codes to prepare the data and train your neural networks, but it is may not compatible with your codes. So, feel free to use your own codes.

In [ ]:
#@title utility functions
# some of the codes are revised from https://github.com/312shan/Pytorch-seq2seq-Beam-Search/blob/master/model.py

def check_gpu():
    # torch.cuda.is_available() checks and returns a Boolean True if a GPU is available, else it'll return False
    is_cuda = torch.cuda.is_available()

    # If we have a GPU available, we'll set our device to GPU. We'll use this device variable later in our code.
    if is_cuda:
        device = torch.device("cuda")
        print("GPU is available")
    else:
        device = torch.device("cpu")
        print("GPU not available, CPU used")
    return device

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def save_model(model, path):
    torch.save(model.state_dict(), path)

def load_model(model, path):
    model.load_state_dict(torch.load(path))
    return model

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.Embedding):
        nn.init.uniform_(m.weight, -0.1, 0.1)
    elif isinstance(m, nn.GRU) or isinstance(m, nn.LSTM) or isinstance(m, nn.RNN):
        for name, param in m.named_parameters():
            if 'weight' in name:
                nn.init.xavier_uniform_(param)
            elif 'bias' in name:
                nn.init.constant_(param, 0)

In [ ]:
#@title training function

def train(model, proc, data, device, translate_func, example_len=15, example_sent="You can come with me."):
    # fetch hyper-parameters
    batch_size = proc.get_model_arg("batch_size")
    teacher_forcing_ratio = proc.get_model_arg('teacher_forcing_ratio')
    clip = proc.get_model_arg('clip')
    n_epochs = proc.get_model_arg('n_epochs')
    learning_rate = proc.get_model_arg("learning_rate")
    verbose = proc.get_model_arg("verbose")
    log_step = proc.get_model_arg("log_step")
    checkpoint_path = proc.get_model_arg("checkpoint_path")

    # prepare training dataset
    features = proc.convert_examples_to_features(data)
    data_iter = proc.get_data_iter(features, batch_size)

    # training steps in each epoch
    examples_total_num = len(data)
    max_steps = math.ceil(float(examples_total_num)/batch_size)

    # Define Loss, Optimizer
    criterion = nn.NLLLoss(ignore_index=proc.pad_token_id)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # train!
    for epoch in range(n_epochs):
        total_loss = 0.0
        model.eval()
        if verbose:
            print( translate_func(model, proc, example_len, example_sent, device) )

        for step in range(max_steps):
            model.train()
            optimizer.zero_grad() # Clears existing gradients from previous epoch
            # prepare inputs
            try:
                batch = next(data_iter)
            except StopIteration:
                data_iter = proc.get_data_iter(features, batch_size)
                batch = next(data_iter)

            # batch[i]: batch_size*seq_len
            src = batch[0].transpose(0, 1).to(device)
            trg = batch[1].transpose(0, 1).to(device)
            # output: seq_len*batch_size*vocab_size
            output = model(src, trg, teacher_forcing_ratio, start_token_id = proc.beg_token_id)

            output_dim = output.shape[-1]
            output = output.transpose(0, 1).contiguous().view(-1, output_dim)

            loss = criterion(output, batch[1].view(-1).to(device))

            loss.backward() # Does backpropagation and calculates gradients
            # clip gradients to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

            optimizer.step() # Updates the weights accordingly

            total_loss += loss.item()
            if verbose and step%log_step==0:
                print("\rstep: {}/{}, Loss: {:.4f}".format(step, max_steps, loss.item()), end=' ')

        print("epoch: {}/{}, Loss: {:.4f}, saving model to {}".format(epoch, n_epochs, total_loss/max_steps, checkpoint_path))
        save_model(model, checkpoint_path)

In [ ]:
#@title Training!
use_att = True
# get parameters from preprocessor
init_seed = proc.get_model_arg('init_seed')
hidden_size = proc.get_model_arg('hidden_size')
n_layers = proc.get_model_arg('n_layers')
dropout = proc.get_model_arg('dropout')

device = check_gpu()
set_seed(init_seed)

# Instantiate the model with hyperparameters
encoder = Encoder(proc.lang1.vocab_size, hidden_size, n_layers, dropout)
# init with or without attention
decoder_cls = Decoder if use_att else VanillaDecoder
decoder = decoder_cls(proc.lang2.vocab_size, hidden_size, n_layers, dropout)
# We'll also set the model to the device that we defined earlier (default is CPU)

model = Seq2Seq(encoder, decoder).to(device)
model.apply(init_weights)

train(model, proc, clean_data, device, greedy_translate)
# train(model, proc, clean_data, device, beam_translate)

In [ ]:
#@title Inference (load well-trained model)!
use_att = True

# saved model for seq2seq with or without attention
# use your own well-trained model
checkpoint_path = "./seq2seq.bin"
arg_path = "./proc.dat"

device = check_gpu()

proc = PreProcessor('en', 'zh')
proc = proc.load(arg_path)
init_seed = proc.get_model_arg('init_seed')
hidden_size = proc.get_model_arg('hidden_size')
n_layers = proc.get_model_arg('n_layers')
dropout = proc.get_model_arg('dropout')
set_seed(init_seed)

# Instantiate the model with hyperparameters
encoder = Encoder(proc.lang1.vocab_size, hidden_size, n_layers, dropout)
# init with or without attention
decoder_cls = Decoder if use_att else VanillaDecoder
decoder = decoder_cls(proc.lang2.vocab_size, hidden_size, n_layers, dropout)

model = Seq2Seq(encoder, decoder).to(device)
model = model.to(device)

model = load_model(model, checkpoint_path)

In [ ]:
greedy_translate(model, proc, 15, "You're a good person.", device)

In [ ]:
proc.set_model_arg('beam_width', 10)
proc.set_model_arg('beam_topk', 4)
beam_translate(model, proc, 15, "You're a good person.", device)